In the CrewAI notebook `First_Multi_Agent.ipynb`, the difference between **Agent** and **Task** is:

**Agent** — The *actor* that performs work. It has a defined **role**, **goal**, and **backstory** that give it a persona and expertise. Examples in the notebook:
- `chef` agent: role="chef", goal="Give 2 different recipes using ingredients", has culinary expertise
- `nutritionist` agent: role="nutritionist", goal="Critically analyze the recipe", has diabetic patient knowledge

**Task** — The *work assignment* that defines **what** needs to be done, the **expected output format**, and **which agent** is responsible. Examples:
- `cook` task: assigned to `chef` agent, asks to write 2 recipes with stepwise guide
- `recommend` task: assigned to `nutritionist` agent, asks to analyze and recommend the best recipe for diabetic patients

**Key distinction**: Agent = *who* does it (the worker with a role/persona). Task = *what* to do (the specific assignment with expected output). The **Crew** orchestrates them together — tasks are assigned to agents, and the crew runs them in sequence, passing outputs between tasks automatically.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
import crewai
from crewai import Agent, Task, Crew

ModuleNotFoundError: No module named 'crewai'

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI/.env_rag")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
import os
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [ ]:
#1- Chef
chef = Agent(
    role="chef",
    goal="Give 2 different receipes using ingredients : {ingredients}",
    backstory="You're  a Chef who has experience on cooking delicious foods"
              "Your expertice lies in recommeding some new foods to the customers"
              "Please generate and give receipe of 2 different foods using this ingredients : {ingredients} "
              "describing stepwise process, time to cook, other ingredients needed and calorie count"
              "Your recipe will be critically analyzed by nutritutionist to recommend food for diabetes patients ",
	verbose=True
)

In [ ]:
2# Nutritionist
nutritionist = Agent(
    role="nutritionist",
    goal="Critically analyze the recipe given by chef using these ingredients: {ingredients}",
    backstory="You're a nutritionist and have good knowledge on working with Diabetic patients "
              "There are recipe recommended by chef using these ingredients : {ingredients}. "
              "please critically analyze the recipe and suggest best food out of given 2 for Diabetic patients"
              "Also give reasons why you dont recommend other 1 receips and choose this one",
    verbose=True
)

In [ ]:
cook = Task(
    description=(
        "1. Write receipe of 2 different foods using these ingredients  : {ingredients}.\n"
    ),
    expected_output="2 different receipes using {ingredients}"
                    "Stepwise guide to cook"
                    "List of all additional ingredients required for cooking"
                    "The expected taste of the food",
    agent=chef,
)

In [ ]:
recommend = Task(
    description=(
        "1. Read the receipes provided by chef using {ingredients} \n"
        "2. critically analyze all receipes keeping in mind what should suit to diabetic patients.\n"
        "3. Suggest one best food out of all 2 receipes given by chef.\n"
        "4. Prove that other 1 receipes suggested by chef is not good for diabetic patients.\n"
        
    ),
    expected_output="A well-written critical pointwise analysis of all 2 recipes using {ingredients}"
                    "Suggest one receipe to diabetic patients based on your own knowledge and evidence "
                    "Reject all other 1 receipes with knowledge and evidence "
                    "Finally, give the stepwoise receipe of what you selected and a nice message",
    agent=nutritionist,
)

In [ ]:
crewaman = Crew(
    agents=[chef,nutritionist],
    tasks=[cook, recommend],
    verbose=2
)

In [ ]:
result = crewaman.kickoff(inputs={"ingredients": "chicken"})